# 00E · Prediction → Planning → Control：一条轨迹如何真正影响车辆？

这三个词在岗位描述里经常连在一起，但它们不是同一个任务：

- **Prediction**：其他交通参与者接下来可能怎么运动？输出不确定的 future distribution。
- **Planning**：在 route、规则、障碍物和舒适性约束下，自车应该采取哪条 trajectory/maneuver？
- **Control**：车辆当前状态如何跟踪这条 trajectory，输出 steering/throttle/brake？

本节用一个前车急刹场景区分 open-loop 与 closed-loop：open-loop 只比较预测/规划与记录答案，closed-loop 则把动作反馈回环境，下一帧输入会因策略而改变。L4 的风险不能只靠单帧 accuracy 描述。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dt = 0.1
horizon = 60
time = np.arange(horizon) * dt
ego_speed = np.full(horizon, 12.0)
lead_speed = np.where(time < 2.0, 10.0, np.maximum(4.0, 10.0 - 3.0 * (time - 2.0)))
gap_open_loop = 25.0 + np.cumsum((lead_speed - ego_speed) * dt)

def safe_speed(gap, lead_speed, reaction_time=0.8, min_gap=5.0):
    available = np.maximum(gap - min_gap, 0.0)
    return np.minimum(lead_speed + available / max(reaction_time, 0.1), 14.0)

planned_speed = safe_speed(gap_open_loop, lead_speed)
closed_loop_gap = 25.0
gaps = [closed_loop_gap]
ego_speed_closed = []
for step in range(horizon - 1):
    command_speed = safe_speed(closed_loop_gap, lead_speed[step])
    ego_speed_closed.append(command_speed)
    closed_loop_gap += (lead_speed[step] - command_speed) * dt
    gaps.append(closed_loop_gap)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(time, ego_speed, label="nominal ego speed")
axes[0].plot(time, lead_speed, label="lead speed")
axes[0].plot(time, planned_speed, label="planner speed")
axes[0].set(title="Planning converts predicted risk to an ego trajectory", xlabel="time / s", ylabel="speed / m/s")
axes[0].legend()
axes[1].plot(time, gap_open_loop, label="open-loop replay")
axes[1].plot(time, gaps, label="closed-loop guarded")
axes[1].axhline(5.0, color="red", linestyle="--", label="minimum gap")
axes[1].set(title="Actions change the next observation", xlabel="time / s", ylabel="gap / m")
axes[1].legend()
plt.tight_layout()


In [ ]:
from ipywidgets import FloatSlider, interact

def closed_loop_experiment(reaction_time=0.8, observation_noise=0.0, horizon_s=6.0):
    steps = int(horizon_s / dt)
    gap = 25.0
    min_gap = 5.0
    rng = np.random.default_rng(42)
    trace = []
    for step in range(steps):
        measured_gap = gap + rng.normal(0, observation_noise)
        command = safe_speed(measured_gap, lead_speed[step], reaction_time=reaction_time, min_gap=min_gap)
        gap += (lead_speed[step] - command) * dt
        trace.append(gap)
    print(f"minimum closed-loop gap={min(trace):.2f} m")
    print("planner/control question:", "fallback or emergency braking" if min(trace) < min_gap else "continue")

interact(
    closed_loop_experiment,
    reaction_time=FloatSlider(min=0.2, max=2.0, step=0.1, value=0.8, description="reaction / s"),
    observation_noise=FloatSlider(min=0, max=2.0, step=0.1, value=0.0, description="gap noise / m"),
    horizon_s=FloatSlider(min=2, max=6, step=0.5, value=6, description="horizon / s"),
)


## 领域检查点

1. 为什么一个预测模型的 ADE/FDE 下降，不一定会让 closed-loop collision rate 下降？
2. planner 输出 trajectory 和 control 输出 steering/brake 的边界在哪里？
3. 哪些约束应该由 learned policy 学习，哪些约束应该由独立 safety layer 兜底？

**下一步**：`08` 进入 prediction metrics，`09` 进入 planning/control，`11` 和 `17` 进入闭环/场景回放。
